In [17]:
import os
from pathlib import Path
import requests
from time import sleep
from setfit import SetFitModel

from mira.sources.sympy_ode.paper_relevance_ranking.utils import (
    download_papers,
    get_pmid_pmc_download_mapping,
    extract_and_get_nxml_paths,
    parse_nxml,
)

# HERE = Path(__file__).parent.resolve()
DATA_PATH = Path("/Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/extracted_papers")
DATA_PATH.mkdir(parents=True, exist_ok=True)

POSITIVE_PATH = DATA_PATH / "positive"
NEGATIVE_PATH = DATA_PATH / "negative"
POSITIVE_PATH.mkdir(parents=True, exist_ok=True)
NEGATIVE_PATH.mkdir(parents=True, exist_ok=True)


def load_setfit_model(model_path=None):
    """
    Load a saved SetFit model.

    Args:
        model_path (str): Path to the saved model (optional)

    Returns:
        SetFitModel: Loaded SetFit model
    """
    if model_path is None:
        model_path = "/Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/pubmed_classifier"
    print(f"📂 Loading SetFit model from: {model_path}")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found at: {model_path}")

    model = SetFitModel.from_pretrained(model_path)
    return model


def extract_pmids_from_pubmed(email, max_results=10000):
    """
    Extract PMIDs from PubMed using the E-utilities API.
    
    Args:
        email (str): Your email (required by NCBI)
        max_results (int): Maximum number of results to retrieve
    
    Returns:
        list: List of PMIDs
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    search_query = '''
    (
        "mathematical model"[Title/Abstract] OR 
        "compartmental model"[Title/Abstract] OR 
        "SEIR"[Title/Abstract] OR "SIR model"[Title/Abstract]
    ) 
    AND 
    (
        "epidemiology"[MeSH Terms] OR 
        epidemiology[sh]
    )'''

    
    print(f"🔍 Searching PubMed with query...")
    
    search_url = f"{base_url}esearch.fcgi"
    params = {
        'db': 'pubmed',
        'term': search_query,
        'retmax': max_results,
        'retmode': 'json',
        'email': email
    }
    
    response = requests.get(search_url, params=params)
    results = response.json()
    
    pmids = results['esearchresult']['idlist']
    total_count = results['esearchresult']['count']
    
    print(f"✅ Found {total_count} total papers, retrieved {len(pmids)} PMIDs")
    
    return pmids


def classify_and_store_papers(pmids, model, pmid_to_download_mapping, 
                               download_path):
        """
        Download, classify, and store papers in positive/negative directories.
        
        Args:
            pmids (list): List of PMIDs to process
            model: Loaded SetFit model
            pmid_to_download_mapping (dict): Mapping from PMID to PMC ID
            download_path (Path): Path to download papers
        Returns:
            dict: Classification results with counts and PMIDs
        """
        positive_pmids = []
        negative_pmids = []
        failed_pmids = []
        try:
            # Download papers
            valid_pmids_for_download = download_papers(pmids, download_path, pmid_to_download_mapping)
            
            # Extract NXML paths
            nxml_paths = extract_and_get_nxml_paths(
                download_path, valid_pmids_for_download, pmid_to_download_mapping
            )
            
            # Parse text content
            texts = []
            valid_pmids = []
            
            for pmid, nxml_path in zip(valid_pmids_for_download, nxml_paths):
                try:
                    if nxml_path and os.path.exists(nxml_path):
                        text = parse_nxml(nxml_path)
                        texts.append(text)
                        valid_pmids.append(pmid)
                    else:
                        failed_pmids.append(pmid)
                except Exception as e:
                    print(f"⚠️  Failed to parse PMID {pmid}: {e}")
                    failed_pmids.append(pmid)
            
            print(texts[:5])

            if not texts:
                print("⚠️  No valid texts in this batch, skipping...")
                # continue
            
            # Classify
            confidences = model.predict_proba(texts)
            predictions = confidences.argmax(dim=1).tolist()

            print(predictions[:20])
            
            # Store results
            for pmid, pred, probs in zip(valid_pmids, predictions, confidences):
                p_neg, p_pos = probs[0].item(), probs[1].item()
                
                if pred == 1:  # Positive
                    positive_pmids.append({
                        'pmid': pmid,
                        'confidence': p_pos
                    })
                    print(f"✅ PMID {pmid}: POSITIVE (conf={p_pos:.3f})")
                else:  # Negative
                    negative_pmids.append({
                        'pmid': pmid,
                        'confidence': p_neg
                    })
                    print(f"❌ PMID {pmid}: NEGATIVE (conf={p_neg:.3f})")
            
            # Rate limiting
            sleep(0.34)
            
        except Exception as e:
            print(f"⚠️  Error processing batch: {e}")
            failed_pmids.extend(pmids)
            # continue
    
        return {
            'positive': positive_pmids,
            'negative': negative_pmids,
            'failed': failed_pmids
        }


def save_results(results, output_path):
    """
    Save classification results to files.
    
    Args:
        results (dict): Classification results
        output_path (Path): Directory to save results
    """
    # Save positive PMIDs
    positive_file = output_path / "positive_pmids.txt"
    with open(positive_file, 'w') as f:
        f.write("PMID\tConfidence\n")
        for item in results['positive']:
            f.write(f"{item['pmid']}\t{item['confidence']:.4f}\n")
    
    # Save negative PMIDs
    negative_file = output_path / "negative_pmids.txt"
    with open(negative_file, 'w') as f:
        f.write("PMID\tConfidence\n")
        for item in results['negative']:
            f.write(f"{item['pmid']}\t{item['confidence']:.4f}\n")
    
    # Save failed PMIDs
    count = 0
    if results['failed']:
        failed_file = output_path / "failed_pmids.txt"
        count+=1
        with open(failed_file, 'w') as f:
            for pmid in results['failed']:
                f.write(f"{pmid}\n")
    
    print(f"\n📊 Results Summary:")
    print(f"   Positive papers: {len(results['positive'])}")
    print(f"   Negative papers: {len(results['negative'])}")
    print(f"   Failed papers: {len(results['failed'])}")
    print(f"\n💾 Results saved to: {output_path}")


def main():
    """Main function to extract, classify, and store papers."""
    
    # Configuration
    EMAIL = "r.mohbe@northeastern.edu"
    MAX_RESULTS = 50  # Adjust as needed
    BATCH_SIZE = 20
    
    # Load model
    print("🚀 Starting pipeline...")
    model = load_setfit_model()
    
    # Get PMID to PMC mapping
    pmid_to_download_mapping = get_pmid_pmc_download_mapping()
    
    # Extract PMIDs from PubMed
    pmids = extract_pmids_from_pubmed(EMAIL, MAX_RESULTS)
    
    if not pmids:
        print("❌ No PMIDs found. Exiting.")
        return
    
    # Classify and store papers
    results = classify_and_store_papers(
        pmids, 
        model, 
        pmid_to_download_mapping,
        DATA_PATH
    )
    
    # Save results
    save_results(results, DATA_PATH)
    
    print("\n✅ Pipeline complete!")


if __name__ == "__main__":
    main()

INFO: [2026-01-28 18:51:00] sentence_transformers.SentenceTransformer - Use pytorch device_name: mps
INFO: [2026-01-28 18:51:00] sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: /Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/pubmed_classifier


🚀 Starting pipeline...
📂 Loading SetFit model from: /Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/pubmed_classifier


Generating PMID to PMC URL mapping: 7533171it [00:12, 605010.76it/s]


🔍 Searching PubMed with query...
✅ Found 5284 total papers, retrieved 50 PMIDs
No PMC mapping found for PMID 41592124, skipping...
No PMC mapping found for PMID 41572568, skipping...
No PMC mapping found for PMID 41537872, skipping...
No PMC mapping found for PMID 41530616, skipping...
No PMC mapping found for PMID 41519155, skipping...
No PMC mapping found for PMID 41511554, skipping...
No PMC mapping found for PMID 41504567, skipping...
No PMC mapping found for PMID 41494848, skipping...
No PMC mapping found for PMID 41494486, skipping...
No PMC mapping found for PMID 41486536, skipping...
No PMC mapping found for PMID 41453653, skipping...
No PMC mapping found for PMID 41431803, skipping...
No PMC mapping found for PMID 41412118, skipping...
No PMC mapping found for PMID 41361035, skipping...
No PMC mapping found for PMID 41354315, skipping...
No PMC mapping found for PMID 41344424, skipping...
No PMC mapping found for PMID 41338380, skipping...
No PMC mapping found for PMID 4132795

In [ ]:
import requests
from time import sleep
import xml.etree.ElementTree as ET


def fetch_abstracts_batch(pmids, email, batch_size=200):
    """
    Fetch titles and abstracts for a batch of PMIDs.
    
    Args:
        pmids (list): List of PMIDs
        email (str): Your email (required by NCBI)
        batch_size (int): Number of PMIDs per request (max 200)
    
    Returns:
        dict: Dictionary mapping PMID to {title, abstract}
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    results = {}
    
    for i in range(0, len(pmids), batch_size):
        batch_pmids = pmids[i:i + batch_size]
        
        params = {
            'db': 'pubmed',
            'id': ','.join(batch_pmids),
            'retmode': 'xml',
            'rettype': 'abstract',
            'email': email
        }
        
        response = requests.get(base_url, params=params)
        
        if response.status_code == 200:
            # Parse XML
            root = ET.fromstring(response.content)
            
            for article in root.findall('.//PubmedArticle'):
                # Extract PMID
                pmid_elem = article.find('.//PMID')
                if pmid_elem is not None:
                    pmid = pmid_elem.text
                    
                    # Extract title
                    title_elem = article.find('.//ArticleTitle')
                    title = title_elem.text if title_elem is not None else ""
                    
                    # Extract abstract (may have multiple sections)
                    abstract_parts = []
                    abstract_texts = article.findall('.//AbstractText')
                    
                    for abs_text in abstract_texts:
                        # Check if abstract has sections (like Background, Methods, etc.)
                        label = abs_text.get('Label', '')
                        text = abs_text.text if abs_text.text else ""
                        
                        if label:
                            abstract_parts.append(f"{label}: {text}")
                        else:
                            abstract_parts.append(text)
                    
                    abstract = " ".join(abstract_parts) if abstract_parts else ""
                    
                    results[pmid] = {
                        'title': title,
                        'abstract': abstract
                    }
        
        # Rate limiting (3 requests per second without API key)
        sleep(0.34)
    
    return results


def fetch_abstracts_simple(pmids, email):
    """
    Simple version using all PMIDs at once (for smaller datasets).
    
    Args:
        pmids (list): List of PMIDs
        email (str): Your email
    
    Returns:
        dict: Dictionary mapping PMID to {title, abstract}
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    
    params = {
        'db': 'pubmed',
        'id': ','.join(pmids),
        'retmode': 'xml',
        'rettype': 'abstract',
        'email': email
    }
    
    response = requests.get(base_url, params=params)
    results = {}
    
    if response.status_code == 200:
        root = ET.fromstring(response.content)
        
        for article in root.findall('.//PubmedArticle'):
            pmid_elem = article.find('.//PMID')
            if pmid_elem is not None:
                pmid = pmid_elem.text
                
                title_elem = article.find('.//ArticleTitle')
                title = title_elem.text if title_elem is not None else ""
                
                abstract_parts = []
                abstract_texts = article.findall('.//AbstractText')
                
                for abs_text in abstract_texts:
                    label = abs_text.get('Label', '')
                    text = abs_text.text if abs_text.text else ""
                    
                    if label:
                        abstract_parts.append(f"{label}: {text}")
                    else:
                        abstract_parts.append(text)
                
                abstract = " ".join(abstract_parts) if abstract_parts else ""
                
                results[pmid] = {
                    'title': title,
                    'abstract': abstract
                }
    
    return results

In [2]:
import os
from pathlib import Path
import requests
from time import sleep
from setfit import SetFitModel
import xml.etree.ElementTree as ET

from mira.sources.sympy_ode.paper_relevance_ranking.utils import (
    download_papers,
    get_pmid_pmc_download_mapping,
    extract_and_get_nxml_paths,
    parse_nxml,
)

# HERE = Path(__file__).parent.resolve()
DATA_PATH = Path("/Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/extracted_papers")
DATA_PATH.mkdir(parents=True, exist_ok=True)

POSITIVE_PATH = DATA_PATH / "positive"
NEGATIVE_PATH = DATA_PATH / "negative"
POSITIVE_PATH.mkdir(parents=True, exist_ok=True)
NEGATIVE_PATH.mkdir(parents=True, exist_ok=True)


def load_setfit_model(model_path=None):
    """
    Load a saved SetFit model.

    Args:
        model_path (str): Path to the saved model (optional)

    Returns:
        SetFitModel: Loaded SetFit model
    """
    if model_path is None:
        model_path = "/Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/pubmed_classifier"
    print(f"📂 Loading SetFit model from: {model_path}")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found at: {model_path}")

    model = SetFitModel.from_pretrained(model_path)
    return model


def extract_pmids_from_pubmed(email, max_results=10000):
    """Extract PMIDs from PubMed using the E-utilities API."""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    
    
    # "mathematical model"[Title/Abstract] OR 
    search_query = '''
    (

        "compartmental model"[Title/Abstract] OR 
        "SEIR"[Title/Abstract] OR 
        "SIR model"[Title/Abstract]
    ) 
    AND 
    (
        "epidemiology"[MeSH Terms] OR 
        epidemiology[sh]
    )'''
    
    print(f"🔍 Searching PubMed...")
    
    search_url = f"{base_url}esearch.fcgi"
    params = {
        'db': 'pubmed',
        'term': search_query,
        'retmax': max_results,
        'retmode': 'json',
        'email': email
    }
    
    response = requests.get(search_url, params=params)
    results = response.json()
    
    pmids = results['esearchresult']['idlist']
    total_count = results['esearchresult']['count']
    
    print(f"✅ Found {total_count} total papers, retrieved {len(pmids)} PMIDs")
    
    return pmids


def fetch_titles_and_abstracts(pmids, email, batch_size=200):
    """
    Fetch titles and abstracts for PMIDs.
    
    Args:
        pmids (list): List of PMIDs
        email (str): Your email
        batch_size (int): PMIDs per request
    
    Returns:
        dict: PMID -> {title, abstract, combined_text}
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    results = {}
    
    total_batches = (len(pmids) + batch_size - 1) // batch_size
    
    for batch_idx, i in enumerate(range(0, len(pmids), batch_size)):
        batch_pmids = pmids[i:i + batch_size]
        
        print(f"📥 Fetching batch {batch_idx + 1}/{total_batches} "
              f"({len(batch_pmids)} papers)...")
        
        params = {
            'db': 'pubmed',
            'id': ','.join(batch_pmids),
            'retmode': 'xml',
            'rettype': 'abstract',
            'email': email
        }
        
        response = requests.get(base_url, params=params)
        
        if response.status_code == 200:
            root = ET.fromstring(response.content)
            
            for article in root.findall('.//PubmedArticle'):
                pmid_elem = article.find('.//PMID')
                if pmid_elem is not None:
                    pmid = pmid_elem.text
                    
                    # Extract title
                    title_elem = article.find('.//ArticleTitle')
                    title = title_elem.text if title_elem is not None else ""
                    
                    # Extract abstract
                    abstract_parts = []
                    abstract_texts = article.findall('.//AbstractText')
                    
                    for abs_text in abstract_texts:
                        label = abs_text.get('Label', '')
                        text = abs_text.text if abs_text.text else ""
                        
                        if label:
                            abstract_parts.append(f"{label}: {text}")
                        else:
                            abstract_parts.append(text)
                    
                    abstract = " ".join(abstract_parts) if abstract_parts else ""
                    
                    # Combine title and abstract for classification
                    combined_text = f"{title} {abstract}".strip()
                    
                    results[pmid] = {
                        'title': title,
                        'abstract': abstract,
                        'combined_text': combined_text
                    }
        
        # Rate limiting
        sleep(0.34)
    
    print(f"✅ Retrieved {len(results)} papers with abstracts")
    return results


def classify_papers(paper_data, model, batch_size=32):
    """
    Classify papers using title + abstract.
    
    Args:
        paper_data (dict): PMID -> {title, abstract, combined_text}
        model: SetFit model
        batch_size (int): Classification batch size
    
    Returns:
        dict: Classification results
    """
    positive_results = []
    negative_results = []
    
    pmids = list(paper_data.keys())
    texts = [paper_data[pmid]['combined_text'] for pmid in pmids]
    
    print(f"\n🔬 Classifying {len(texts)} papers...")
    
    # Classify in batches
    all_predictions = []
    all_confidences = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        confidences = model.predict_proba(batch_texts)
        predictions = confidences.argmax(dim=1).tolist()
        
        all_predictions.extend(predictions)
        all_confidences.extend(confidences)
    
    # Store results
    for pmid, pred, probs in zip(pmids, all_predictions, all_confidences):
        p_neg, p_pos = probs[0].item(), probs[1].item()
        
        result = {
            'pmid': pmid,
            'title': paper_data[pmid]['title'],
            'abstract': paper_data[pmid]['abstract'],
            'confidence': p_pos if pred == 1 else p_neg,
            'prob_positive': p_pos,
            'prob_negative': p_neg
        }
        
        if pred == 1:
            positive_results.append(result)
            print(f"✅ PMID {pmid}: POSITIVE (conf={p_pos:.3f})")
        else:
            negative_results.append(result)
            print(f"❌ PMID {pmid}: NEGATIVE (conf={p_neg:.3f})")
    
    return {
        'positive': positive_results,
        'negative': negative_results
    }


def save_results(results, output_path):
    """Save classification results with titles and abstracts."""
    # Save positive papers
    positive_file = output_path / "positive_papers.tsv"
    with open(positive_file, 'w', encoding='utf-8') as f:
        f.write("PMID\tConfidence\tProb_Pos\tProb_Neg\tTitle\tAbstract\n")
        for item in results['positive']:
            f.write(f"{item['pmid']}\t{item['confidence']:.4f}\t"
                   f"{item['prob_positive']:.4f}\t{item['prob_negative']:.4f}\t"
                   f"{item['title']}\t{item['abstract']}\n")
    
    # Save negative papers
    negative_file = output_path / "negative_papers.tsv"
    with open(negative_file, 'w', encoding='utf-8') as f:
        f.write("PMID\tConfidence\tProb_Pos\tProb_Neg\tTitle\tAbstract\n")
        for item in results['negative']:
            f.write(f"{item['pmid']}\t{item['confidence']:.4f}\t"
                   f"{item['prob_positive']:.4f}\t{item['prob_negative']:.4f}\t"
                   f"{item['title']}\t{item['abstract']}\n")
    
    print(f"\n📊 Results Summary:")
    print(f"   Positive papers: {len(results['positive'])}")
    print(f"   Negative papers: {len(results['negative'])}")
    print(f"\n💾 Results saved to: {output_path}")


def main():
    """Main pipeline using titles and abstracts."""
    
    # Configuration
    EMAIL = "r.mohbe@northeastern.edu"
    MAX_RESULTS = 1000
    
    # Load model
    print("🚀 Starting pipeline...")
    model = load_setfit_model()
    
    # Extract PMIDs
    pmids = extract_pmids_from_pubmed(EMAIL, MAX_RESULTS)
    
    if not pmids:
        print("❌ No PMIDs found. Exiting.")
        return
    
    # Fetch titles and abstracts
    paper_data = fetch_titles_and_abstracts(pmids, EMAIL)
    
    if not paper_data:
        print("❌ No papers retrieved. Exiting.")
        return
    
    # Classify papers
    results = classify_papers(paper_data, model)
    
    # Save results
    save_results(results, DATA_PATH)
    
    print("\n✅ Pipeline complete!")


if __name__ == "__main__":
    main()

INFO: [2026-01-29 13:20:45] sentence_transformers.SentenceTransformer - Use pytorch device_name: mps
INFO: [2026-01-29 13:20:45] sentence_transformers.SentenceTransformer - Load pretrained SentenceTransformer: /Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/pubmed_classifier


🚀 Starting pipeline...
📂 Loading SetFit model from: /Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/pubmed_classifier
🔍 Searching PubMed...
✅ Found 1894 total papers, retrieved 1000 PMIDs
📥 Fetching batch 1/5 (200 papers)...
📥 Fetching batch 2/5 (200 papers)...
📥 Fetching batch 3/5 (200 papers)...
📥 Fetching batch 4/5 (200 papers)...
📥 Fetching batch 5/5 (200 papers)...
✅ Retrieved 1000 papers with abstracts

🔬 Classifying 1000 papers...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

❌ PMID 41544184: NEGATIVE (conf=0.838)
❌ PMID 41537872: NEGATIVE (conf=0.815)
❌ PMID 41504567: NEGATIVE (conf=0.991)
❌ PMID 41494848: NEGATIVE (conf=0.818)
❌ PMID 41494486: NEGATIVE (conf=0.743)
✅ PMID 41453653: POSITIVE (conf=0.721)
❌ PMID 41413153: NEGATIVE (conf=0.932)
❌ PMID 41398247: NEGATIVE (conf=0.983)
❌ PMID 41366273: NEGATIVE (conf=0.663)
✅ PMID 41361573: POSITIVE (conf=0.759)
✅ PMID 41361035: POSITIVE (conf=0.621)
❌ PMID 41351019: NEGATIVE (conf=0.753)
✅ PMID 41344424: POSITIVE (conf=0.662)
❌ PMID 41338380: NEGATIVE (conf=0.538)
✅ PMID 41327944: POSITIVE (conf=0.812)
❌ PMID 41318717: NEGATIVE (conf=0.845)
❌ PMID 41298851: NEGATIVE (conf=0.801)
❌ PMID 41290140: NEGATIVE (conf=0.966)
✅ PMID 41258492: POSITIVE (conf=0.852)
✅ PMID 41257909: POSITIVE (conf=0.813)
✅ PMID 41239586: POSITIVE (conf=0.951)
✅ PMID 41224308: POSITIVE (conf=0.889)
❌ PMID 41213282: NEGATIVE (conf=0.953)
❌ PMID 41188786: NEGATIVE (conf=0.520)
❌ PMID 41188230: NEGATIVE (conf=0.703)
❌ PMID 41184682: NEGATIVE

In [17]:
import pandas as pd
import pystow
import tqdm as tqdm
from pathlib import Path
from mira.sources.sympy_ode.paper_extraction import get_template_model_from_pmid
from mira.modeling import Model
from mira.modeling.ode import OdeModel
from IPython.display import Image
from indra.literature.pubmed_client import (
    download_package_for_pmid,
    get_pmid_to_package_url_mapping,
    pmid_to_pmc_download_url,
)

DATA_PATH = Path("/Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/extracted_papers")
DATA_PATH.mkdir(parents=True, exist_ok=True)

POSITIVE_PATH = DATA_PATH / "positive"
NEGATIVE_PATH = DATA_PATH / "negative"
POSITIVE_PATH.mkdir(parents=True, exist_ok=True)
NEGATIVE_PATH.mkdir(parents=True, exist_ok=True)

positive_papers = DATA_PATH / "positive_papers.tsv"
df = pd.read_csv(positive_papers, sep='\t')

filepath = DATA_PATH / "extracted_ode_strs.csv"
extracted_odes={}

def get_pmid_to_pmc_mapping_path() -> Path:
    return pystow.ensure(
        "mira", "paper_extraction", url=pmid_to_pmc_download_url
    )

pmid_to_download_mapping = get_pmid_to_package_url_mapping(
        get_pmid_to_pmc_mapping_path().as_posix()
    )

for idx, row in tqdm.tqdm(df.iterrows()):
    pmid = str(row["PMID"])
    try:
        # pmid = "33869905"
        print(f"Processing PMID {pmid}...")
        tm, ode_str = get_template_model_from_pmid(pmid, "image", pmid_to_download_mapping)
        print(f"\n🔬 PMID {pmid} Template Model:\n{ode_str}\n")
        extracted_odes[pmid] = ode_str
        print("concept name\tidentifiers\tcontext")
        for concept in tm.get_concepts_map().values():
            print(f"{concept.name}\t{concept.identifiers}\t{concept.context}")
        om = OdeModel(Model(tm), initialized=True)
        om.get_interpretable_kinetics()
    except Exception as e:
        print(f"⚠️  Failed to extract model for PMID {pmid}: {e} \n")
        continue
    


Generating PMID to PMC URL mapping: 7533171it [00:12, 588292.32it/s]
0it [00:00, ?it/s]INFO: [2026-02-02 10:38:31] indra.literature.pubmed_client - Downloading https://ftp.ncbi.nlm.nih.gov/pub/pmc/oa_package/b0/bc/PMC12800035.tar.gz


Processing PMID 41453653...
⚠️  Failed to extract model for PMID 41453653: PMID 41453653 not found in the PMC OA mapping. 

Processing PMID 41361573...


2026-02-02 10:38:40.370 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze:128 - Batch 1/1: 17 pages/17 pages
OCR-rec Predict: 100%|██████████| 20/20 [00:01<00:00, 18.50it/s]
2026-02-02 10:40:13.058 | INFO     | mineru.cli.common:_process_output:151 - local output dir is /var/folders/4t/2mddk68964jght7vrv17hw8m0000gq/T/tmpuse0paxv/41598_2025_Article_31376/auto
INFO: [2026-02-02 10:40:13] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 10:40:13] mira.sources.sympy_ode.agent_pipeline - MULTI-AGENT ODE EXTRACTION & VALIDATION PIPELINE
INFO: [2026-02-02 10:40:13] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 10:40:13] mira.sources.sympy_ode.agent_pipeline - PHASE 1: ODE Extraction from input
INFO: [2026-02-02 10:40:31] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2026-02-02 10:40:31] 


🔬 PMID 41361573 Template Model:
# Define time variable
t = sympy.symbols("t")

# Define the time-dependent variables
S1, S2, I1, I2, A = sympy.symbols("S1 S2 I1 I2 A", cls=sympy.Function)

# Define the parameters
eta, b1, b2, b3, b4, lambda_, mu, theta, d = sympy.symbols("eta b1 b2 b3 b4 lambda mu theta d")

odes = [
    sympy.Eq(S1(t).diff(t), (1 - theta) * eta - (b1 * S2(t) - (lambda_ + mu) * S1(t))),
    sympy.Eq(S2(t).diff(t), theta * eta - (lambda_ + b1 + mu) * S2(t)),
    sympy.Eq(I1(t).diff(t), lambda_ * S1(t) + b2 * I2(t) - (b3 + mu) * I1(t)),
    sympy.Eq(I2(t).diff(t), lambda_ * S2(t) - (b2 + b4 + mu) * I2(t)),
    sympy.Eq(A(t).diff(t), b3 * I1(t) + b4 * I2(t) - (mu + d) * A(t))
]

concept name	identifiers	context
A	{'ido': '0000511'}	{'stage': 'asymptomatic', 'species': 'ncbitaxon:9606'}
I1	{'ido': '0000511'}	{'disease_severity': 'ncit:C25269'}
S1	{'ido': '0000514'}	{'species': 'ncbitaxon:9606'}
Processing PMID 41361035...
⚠️  Failed to extract model for PMID 41361035: PMI

2026-02-02 10:41:03.983 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze:128 - Batch 1/1: 14 pages/14 pages
OCR-rec Predict: 100%|██████████| 19/19 [00:01<00:00, 17.26it/s]
2026-02-02 10:43:12.736 | INFO     | mineru.cli.common:_process_output:151 - local output dir is /var/folders/4t/2mddk68964jght7vrv17hw8m0000gq/T/tmp7o9inxc1/41598_2025_Article_24606/auto
INFO: [2026-02-02 10:43:12] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 10:43:12] mira.sources.sympy_ode.agent_pipeline - MULTI-AGENT ODE EXTRACTION & VALIDATION PIPELINE
INFO: [2026-02-02 10:43:12] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 10:43:12] mira.sources.sympy_ode.agent_pipeline - PHASE 1: ODE Extraction from input
INFO: [2026-02-02 10:43:35] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2026-02-02 10:43:35] 

...


🔬 PMID 40641244 Template Model:
import sympy

# Define time variable and state functions
t = sympy.symbols("t")
S = sympy.Function("S")(t)

# Define parameters
beta = sympy.symbols("beta")

# Define the variables
w, a, v = sympy.symbols("w a v")

# Define the ACH equation as an ODE
ACH = 0.65 * w * a * 3600 / v
odes = [sympy.Eq(S.diff(t), ACH)]

concept name	identifiers	context
S	{'ido': '0000514'}	{}
Processing PMID 40633683...
⚠️  Failed to extract model for PMID 40633683: PMID 40633683 not found in the PMC OA mapping. 

Processing PMID 40450593...


2026-02-02 11:29:18.986 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze:128 - Batch 1/1: 9 pages/9 pages
OCR-rec Predict: 100%|██████████| 7/7 [00:04<00:00,  1.68it/s]
2026-02-02 11:30:19.203 | INFO     | mineru.cli.common:_process_output:151 - local output dir is /var/folders/4t/2mddk68964jght7vrv17hw8m0000gq/T/tmp6q2i8vyz/MJA2-222-558/auto
INFO: [2026-02-02 11:30:19] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 11:30:19] mira.sources.sympy_ode.agent_pipeline - MULTI-AGENT ODE EXTRACTION & VALIDATION PIPELINE
INFO: [2026-02-02 11:30:19] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 11:30:19] mira.sources.sympy_ode.agent_pipeline - PHASE 1: ODE Extraction from input
INFO: [2026-02-02 11:30:26] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2026-02-02 11:30:26] mira.sources.sym


🔬 PMID 40450593 Template Model:
# Define time variable
t = sympy.symbols("t")

# Define the time-dependent variables
S, I, R = sympy.symbols("S I R", cls=sympy.Function)

# Define the parameters
b, g = sympy.symbols("b g")

odes = [
    sympy.Eq(S(t).diff(t), - b * S(t) * I(t)),
    sympy.Eq(I(t).diff(t), b * S(t) * I(t) - g * I(t)),
    sympy.Eq(R(t).diff(t), g * I(t))
]

concept name	identifiers	context
S	{'ido': '0000514'}	{'severity': 'susceptible', 'species': 'ncbitaxon:9606'}
I	{'ido': '0000511'}	{'disease_severity': 'ncit:C25269'}
R	{'ido': '0000592'}	{'species': 'ncbitaxon:9606'}
Processing PMID 40449318...
⚠️  Failed to extract model for PMID 40449318: PMID 40449318 not found in the PMC OA mapping. 

Processing PMID 40435153...


2026-02-02 11:30:56.244 | INFO     | mineru.backend.pipeline.pipeline_analyze:doc_analyze:128 - Batch 1/1: 25 pages/25 pages
OCR-rec Predict: 100%|██████████| 2/2 [00:00<00:00,  8.60it/s]
2026-02-02 11:34:16.304 | INFO     | mineru.cli.common:_process_output:151 - local output dir is /var/folders/4t/2mddk68964jght7vrv17hw8m0000gq/T/tmpvy6chpmb/pone.0321089/auto
INFO: [2026-02-02 11:34:16] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 11:34:16] mira.sources.sympy_ode.agent_pipeline - MULTI-AGENT ODE EXTRACTION & VALIDATION PIPELINE
INFO: [2026-02-02 11:34:16] mira.sources.sympy_ode.agent_pipeline - ------------------------------------------------------------
INFO: [2026-02-02 11:34:16] mira.sources.sympy_ode.agent_pipeline - PHASE 1: ODE Extraction from input
INFO: [2026-02-02 11:34:40] httpx - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: [2026-02-02 11:34:40] mira.sources.s

KeyboardInterrupt: 

In [21]:
idx, len(extracted_odes)

(37, 16)

In [18]:
extracted_odes

{'41361573': '# Define time variable\nt = sympy.symbols("t")\n\n# Define the time-dependent variables\nS1, S2, I1, I2, A = sympy.symbols("S1 S2 I1 I2 A", cls=sympy.Function)\n\n# Define the parameters\neta, b1, b2, b3, b4, lambda_, mu, theta, d = sympy.symbols("eta b1 b2 b3 b4 lambda mu theta d")\n\nodes = [\n    sympy.Eq(S1(t).diff(t), (1 - theta) * eta - (b1 * S2(t) - (lambda_ + mu) * S1(t))),\n    sympy.Eq(S2(t).diff(t), theta * eta - (lambda_ + b1 + mu) * S2(t)),\n    sympy.Eq(I1(t).diff(t), lambda_ * S1(t) + b2 * I2(t) - (b3 + mu) * I1(t)),\n    sympy.Eq(I2(t).diff(t), lambda_ * S2(t) - (b2 + b4 + mu) * I2(t)),\n    sympy.Eq(A(t).diff(t), b3 * I1(t) + b4 * I2(t) - (mu + d) * A(t))\n]',
 '41239586': '# Define time variable\nt = sympy.symbols("t")\n\n# Define the time-dependent variables\nS, E, I, R = sympy.symbols("S E I R", cls=sympy.Function)\n\n# Define the parameters\nbeta, omega, gamma = sympy.symbols("beta omega gamma")\n\nodes = [\n    sympy.Eq(S(t).diff(t), - beta * S(t) * 

In [ ]:
# 📊 Results Summary:
#    Positive papers: 443
#    Negative papers: 557

# 💾 Results saved to: /Users/mohbe.r/Documents/CODE/NEU/mira/mira/sources/sympy_ode/paper_relevance_ranking/extracted_papers

# ✅ Pipeline complete!